# Sentiment dataset: k-NN

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from models.knn import KNN
from utils.data import load_train
from utils.evaluation import evaluate, print_report, get_confusion_matrix
from utils.preprocessing import make_preprocessor

X, y = load_train("sentiment")
X, y = np.asarray(X), np.asarray(y)

# 80% to train on, 20% to validate on
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " Validation:", X_val.shape,
      " Classes:", len(np.unique(y)))

Train: (9600, 768)  Validation: (2400, 768)  Classes: 3


## Preprocessing

Scaling the features before measuring distances

In [3]:
rows = []
best_score = 0

for preprocessing in ["none", "standard", "l2"]:
    prep = make_preprocessor(preprocessing)
    Xtr = prep.fit_transform(X_train, y_train)
    Xv = prep.transform(X_val)

    model = KNN(k=5, distance="euclidean")
    model.fit(Xtr, y_train)

    metrics = evaluate(y_val, model.predict(Xv))
    rows.append({"preprocessing": preprocessing, **metrics})

    if metrics["accuracy"] > best_score:
        best_score, best_preprocessing = metrics["accuracy"], preprocessing

display(pd.DataFrame(rows))
print("Kept preprocessing =", best_preprocessing)

# Every later step uses the data prepared with the best preprocessing
prep = make_preprocessor(best_preprocessing)
Xtr = prep.fit_transform(X_train, y_train)
Xv = prep.transform(X_val)

,preprocessing,accuracy,macro_f1
0,none,0.538750,0.531231
1,standard,0.540833,0.532868
2,l2,0.546667,0.540283


Kept preprocessing = l2


## Number of neighbours (k)

How many nearest training examples vote on the label

In [4]:
rows = []
best_score = 0

for k in [1, 5, 15, 31, 51, 101]:
    model = KNN(k=k, distance="euclidean")
    model.fit(Xtr, y_train)

    metrics = evaluate(y_val, model.predict(Xv))
    rows.append({"k": k, **metrics})

    if metrics["accuracy"] > best_score:
        best_score, best_k = metrics["accuracy"], k

display(pd.DataFrame(rows))
print("Kept k =", best_k)

,k,accuracy,macro_f1
0,1,0.511667,0.509969
1,5,0.546667,0.540283
2,15,0.559167,0.549486
3,31,0.579583,0.566280
4,51,0.574583,0.558956
5,101,0.578333,0.556572


Kept k = 31


## Distance

How closeness between two examples is measured. The density penalty in the next step only works with cosine.

In [5]:
rows = []

for distance in ["euclidean", "cosine"]:
    model = KNN(k=best_k, distance=distance)
    model.fit(Xtr, y_train)

    metrics = evaluate(y_val, model.predict(Xv))
    rows.append({"distance": distance, **metrics})

display(pd.DataFrame(rows))

,distance,accuracy,macro_f1
0,euclidean,0.579583,0.56628
1,cosine,0.579583,0.56628


## Density penalty

Lowers the score of training examples that look similar to everything, so they stop winning by default

In [6]:
rows = []
best_score = 0

for density_penalty in [0, 0.3, 0.5, 0.7, 1.0]:
    model = KNN(k=best_k, distance="cosine",
                density_penalty=density_penalty, density_neighbors=20)
    model.fit(Xtr, y_train)

    metrics = evaluate(y_val, model.predict(Xv))
    rows.append({"density_penalty": density_penalty, **metrics})

    if metrics["accuracy"] > best_score:
        best_score, best_density_penalty = metrics["accuracy"], density_penalty

display(pd.DataFrame(rows))
print("Kept density_penalty =", best_density_penalty)

,density_penalty,accuracy,macro_f1
0,0.0,0.579583,0.566280
1,0.3,0.582083,0.571981
2,0.5,0.581667,0.574499
3,0.7,0.578333,0.572543
4,1.0,0.577500,0.575360


Kept density_penalty = 0.3


## Density neighbours

How many neighbours are used to measure how crowded each training example is

In [7]:
rows = []
best_score = 0

for density_neighbors in [10, 20, 50]:
    model = KNN(k=best_k, distance="cosine",
                density_penalty=best_density_penalty,
                density_neighbors=density_neighbors)
    model.fit(Xtr, y_train)

    metrics = evaluate(y_val, model.predict(Xv))
    rows.append({"density_neighbors": density_neighbors, **metrics})

    if metrics["accuracy"] > best_score:
        best_score, best_density_neighbors = metrics["accuracy"], density_neighbors

display(pd.DataFrame(rows))
print("Kept density_neighbors =", best_density_neighbors)

,density_neighbors,accuracy,macro_f1
0,10,0.583750,0.573623
1,20,0.582083,0.571981
2,50,0.582083,0.572635


Kept density_neighbors = 10


## Final Model

The hyperparameters we reached and their scores.

In [8]:
print("preprocessing:", best_preprocessing, " k:", best_k, " distance: cosine",
      " density_penalty:", best_density_penalty,
      " density_neighbors:", best_density_neighbors)

model = KNN(k=best_k, distance="cosine",
            density_penalty=best_density_penalty,
            density_neighbors=best_density_neighbors)
model.fit(Xtr, y_train)

val_predictions = model.predict(Xv)
print(evaluate(y_val, val_predictions))
print_report(y_val, val_predictions)

labels = np.unique(y_val)
matrix = get_confusion_matrix(y_val, val_predictions, labels=labels)
display(pd.DataFrame(matrix,
                     index=[f"true {label}" for label in labels],
                     columns=[f"predicted {label}" for label in labels]))

preprocessing: l2  k: 31  distance: cosine  density_penalty: 0.3  density_neighbors: 10
{'accuracy': 0.58375, 'macro_f1': 0.5736233822069167}
              precision    recall  f1-score   support

           0       0.58      0.73      0.65       800
           1       0.56      0.38      0.45       800
           2       0.61      0.64      0.62       800

    accuracy                           0.58      2400
   macro avg       0.58      0.58      0.57      2400
weighted avg       0.58      0.58      0.57      2400



,predicted 0,predicted 1,predicted 2
true 0,587,97,116
true 1,282,304,214
true 2,144,146,510
